# Developpement du modèle pytorch YOLO pour enderscope

# 1. Installation des différentes bibliothèques et mise à jour 

### &emsp;&emsp;&emsp;&emsp; 1.1) Mise à jour de la fonction d'installation de python pip

In [ ]:
!python -m pip install --upgrade pip

### &emsp;&emsp;&emsp;&emsp;1.2) Installation des différentes bibliothèques nécéssaires :

nous allons installer matplotlib, ultralytics et pytorch cuda (pour l'accélération graphique) attention à votre version de cuda 
pour la connaitre tapper sous une invite de commande  : </p>
<i><b>nvcc --version</i></b> </p>
puis dans la troisième cellule </p>
install torch torchvision --index-url https://download.pytorch.org/whl/<i><b>cu112</i></b> remplacer <i><b>cu112</i></b> par votre<i><b> version</i></b>

In [ ]:
!pip install opencv-python 

In [ ]:
!pip install matplotlib

<i><b>Attention</i></b> </p>

La cellule suivante n'est qu'un exemple (ici python 3.13.15 avec un Cuda 13.2 installé) </p>
En fonction de votre installation python et Cuda veuillez vous reporter sur le lien suivant </p>

[Documentation PyTorch](https://pytorch.org)


In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [ ]:
!pip install ultralytics==8.3.70

la cellule suivante permet de verifier si torch utilise le cuda et si vottre carte graphique est bien prise ebn compte 

In [ ]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu132

In [ ]:
import sys
import os

print("Python :", sys.version)
print("Executable :", sys.executable)
print("Répertoire courant :", os.getcwd())


In [ ]:
import torch

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
print("Version CUDA PyTorch :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


# 2. Mise en place de l'espace de travail 

 ### &emsp;&emsp;&emsp;&emsp;2.1) création de la variable de l'espace de travail 

In [ ]:
# on va stocker le chemin du repertoire de travail dans la variable WORK_PATH
WORK_PATH=%pwd
#comme nous somme sur windows le chemin stocké est de forme : D:\folder\files
#pour que cela fonctionne en python le chemin doit être de la forme D:/folder/files 
# pour cela on utilise la fonction replace
WORK_PATH=WORK_PATH.replace("\\","/")
print(WORK_PATH)

### &emsp;&emsp;&emsp;&emsp;2.2) création du dossier images  

In [ ]:
#on creer le dossier images 
import os
IMAGES_PATH = os.path.join(WORK_PATH,'images')
IMAGES_PATH=IMAGES_PATH.replace("\\","/")

## si le dossiers n'existent pas
if not os.path.exists(IMAGES_PATH):
    
    # si l'OS est Linux 
    if os.name == 'posix':
        # on crée les dossiers Tensorflow , workspace, images, et collectedimages 
        !mkdir -p {IMAGES_PATH}
        
    #si l'OS est windows
    if os.name == 'nt':
        # on crée les dossiers Tensorflow , workspace, images, et collectedimages
        !mkdir "{IMAGES_PATH}"
        
print("dossier crée")

### &emsp;&emsp;&emsp;&emsp;2.3) copier vos images dans le dossier crée </p>

copier toutes vos images dans ce dossier ou alors vous pouvez utiliser la cellule suivante </p>
si vous ne voulez pas faire un copier coller  

In [ ]:
# on installe la bibliothèque ipyfilechooser permettant dans jupyter notebook de choisir le dossier ou sont stocker les images 
# le mieux est d'utiliser des images en 640x480
!pip install ipyfilechooser==0.5.0

# on importe la bibliothèque ipyfilechooser
from ipyfilechooser import FileChooser

fc = FileChooser('')
#on affiche dans jupyter notebook un bouton select
#(appuyer dessus et choisir la première image du dossier ou sont contenu toutes vos images et cliquez sur change
display(fc)

In [ ]:
# on va renomer les images importés  avec un numéro unique en se servant openCV
# on importe la bibliothèque glob permettant la recherche de chemin 
import glob, shutil

# pour toutes les images du dossier selectionné dans la cellule précédente
for img in glob.glob(fc.selected_path+"/*.jpg"):
    shutil.copy(img, IMAGES_PATH)
    
print("Done")

In [ ]:
# on importe uuid pour Universal Unique Identifier, bibliothèque permettant d'assigner  un nombre de 128 bits  pour identifier individuellement 
# les données dans les systèmes informatiques dans notre cas les images que nous allons sauvegarder.
import uuid

#on va renommer toutes les images avec un numéro unique attention changer l'extention si cette dernière ne correspond pas (".jpg")
for filename in os.listdir(IMAGES_PATH):
    if filename.endswith(".jpg"):
        dst=(IMAGES_PATH+'/'+'{}.jpg'.format(str(uuid.uuid1())))
        src=f"{IMAGES_PATH}/{filename}"
        os.rename(src,dst)
print("DONE")

# 3. Annotations des images

### &emsp;&emsp;&emsp;&emsp;3.1)Installation du logiciel LabelImg pour annoter les images </p>

In [ ]:
!pip install --upgrade pyqt5 lxml

In [ ]:
import os
LABELIMG_PATH = os.path.join('Label', 'labelimg')

In [ ]:
if not os.path.exists(LABELIMG_PATH):
    !mkdir {LABELIMG_PATH}
    !git clone https://github.com/tzutalin/labelImg {LABELIMG_PATH}

In [ ]:
if os.name == 'posix':
    !make qt5py3
if os.name =='nt':
    !cd {LABELIMG_PATH} && pyrcc5 -o libs/resources.py resources.qrc

### &emsp;&emsp;&emsp;&emsp;3.2) creer fichier texte de vos classes </p>
on va creer un fichier texte de vos classes, ce fichier remplacera dans LabelImg les classes par default

In [ ]:
contenu = "droite\ngauche\ndouble"

with open("predefined_classes.txt", "w") as f:
    f.write(contenu)

on va maintenant deplacer le fichier texte crée dans le dossier spécifique de LabelImg

In [ ]:
import shutil
import os

source=os.path.join(WORK_PATH,'predefined_classes.txt')
source=source.replace("\\","/")

destination=os.path.join(WORK_PATH,LABELIMG_PATH,'data')
destination=destination.replace("\\","/")

fichier_destination=os.path.join(destination,'predefined_classes.txt')
fichier_destination=fichier_destination.replace("\\","/")



# Si le fichier_destination existe deja on le supprime  
if os.path.exists(fichier_destination):
    os.remove(fichier_destination)

shutil.move(source, destination)

print("Fichier copié")

### &emsp;&emsp;&emsp;&emsp;c) annoter toutes les images avec Labelimg </p>
</p> annoter vos zones avec une box, lui donner un nom et sauvegarder sous format yolo suivez le protocole suivant :</p>
- Cliquer sur Open Dir pou choisir le dossier contenant les images
<img src="pictures/LbelImg_open_dir.PNG" width="90">
- Cliquer ensuite sur Creta bos pour definir une boîte autour de votre élement
<img src="pictures/LbelImg_box.PNG" width="90">
- Donner un nom à votre classe ou élément (si vous avez remplacer le fichier "predefined_classes.txt" vos classes apparaissent sur la droite de l'écran
- Cliquer ensuite sur l'icone PascalVoC pour la remplacer par YOLO
<img src="pictures/LbelImg_pascal.PNG" width="90"><img src="pictures/LbelImg_yolo.PNG" width="90">
- Cliquer ensuite sur save
<img src="pictures/LbelImg_save_dir.PNG">
- Cliquer ensuite sur next image pour ouvrir l'image suivante 
<img src="pictures/LbelImg_next_image.PNG"> 

cela génerera un fichier .txt avec le nom de votre image. Le fichier txt est de type :</p>
1 0.407813 0.635417 0.212500 0.287500 correspondant à class_id x_center y_center width height </p>

<strong>donc bien faire attention à l'id et l'ordre de la class dans LabelImg </strong>

In [ ]:
!cd {LABELIMG_PATH} && python labelImg.py

# 4. organisation de l'environnement de travail pytoch
### &emsp;&emsp;&emsp;&emsp;4.1) Creation des dossiers pour le fonctionnement de PyTorch </p>

on va créer dans la cellule suivante les dossiers ou repartir les images et les annotations </p> selon l'architecture 
de pytorch : il faut absolument disposer les dossiesr ainsi :</p> 
<img src="pictures/architecture.PNG" width="200">

In [ ]:
#on va créer les trois dossiers de repartitions des images 
train_images=os.path.join(WORK_PATH,'Images','train_images')
train_images=train_images.replace("\\","/")
val_images=os.path.join(WORK_PATH,'Images','val_images')
val_images=val_images.replace("\\","/")
test_images=os.path.join(WORK_PATH,'Images','test_images')
test_images=test_images.replace("\\","/")

folders = [train_images,val_images,test_images]

#on créer les dossiers correspondant pour les images 
for nom in folders:
    os.makedirs(nom,exist_ok=True)
print ("Dossiers Images créer")

#on va créer les trois dossier de repartition des labels 
train_labels=os.path.join(WORK_PATH,'labels','train_images')
train_labels=train_labels.replace("\\","/")
val_labels=os.path.join(WORK_PATH,'labels','val_images')
val_labels=val_labels.replace("\\","/")
test_labels=os.path.join(WORK_PATH,'labels','test_images')
test_labels=test_labels.replace("\\","/")

folders = [train_labels,val_labels,test_labels]

#on créer les dossiers correspondant pour les labels 
for nom in folders:
    os.makedirs(nom,exist_ok=True)
print ("Dossiers Labels créer")


### &emsp;&emsp;&emsp;&emsp;4.2) Répartition aléatoire des images dans les dossiers pour Tensorflow </p>

nous allons ensuite diviser aléatoirement les images de chaque labels dans les dossiers train,  validation et  test. Voici à quoi sert chaque ensemble :

* **Train** : Il s'agit des images utilisées pour entraîner le modèle. À chaque étape de l'entrainement, un lot d'images du jeu d'entrainement est tirées alléatoirement pour entrainer le réseau. Le réseau prédit les classes et les emplacements des objets dans les images. L'algorithme d'optimisation calcule la perte (c'est-à-dire le degré d'« erreur » des prédictions) et ajuste les poids du réseau par rétropropagation.

* **Validation** : Les images du dossier validation peuvent être utilisées par l'algorithme d'apprentissage pour vérifier la progression de l'apprentissage et ajuster les hyperparamètres. Contrairement aux images du dossier train, ces images ne sont utilisées que périodiquement au cours de l'entrainement.

* **Test** : Ces images ne sont jamais vues par le réseau pendant l'entrainement. Elles sont destinées à être utilisées par un humain pour effectuer un test final du modèle afin d'en vérifier la précision.

Pour répartir aléatoirement les images de chaque dossier nous allons lancer le script suivant qui va déplacer aléatoirement :</p>
80 % des images vers le dossier train</p> 
10 % vers le dossier validation</p> 
10 % vers le dossier test 

In [ ]:
# on importe les biblothèques nécéssaires 
from pathlib import Path
import random
import os
import sys
import shutil

#on stocke les variables nécésssaires à savoir ou sont les images et le dossier Labels
IMAGES_PATH=os.path.join(WORK_PATH,"images")
LABELS_PATH=os.path.join(WORK_PATH,"labels")

#les variables de sorties on dejà été définie dans la cellule précédentes on les imprime pour être sur mais 
#vous pouvez supprimer les 7 lignes suivantes qui ne sont pas nécéssaire
print("variables:")
print(train_images)
print(val_images)
print(test_images)
print(train_labels)
print(val_labels)
print(test_labels)

# --- Extensions d’images acceptées ---
EXTS = [".jpg", ".jpeg", ".png", ".bmp"]


# --- Récupération de toutes les images du dossier images/ ---
all_images = [os.path.join(IMAGES_PATH, f) for f in os.listdir(IMAGES_PATH) if os.path.splitext(f)[1].lower() in EXTS]

 # Mélanger les images de manière aléatoire
random.shuffle(all_images)

 # Calculer les indices de découpage pour 80%, 10%, 10%
train_size = round((0.8 * len(all_images)))
val_size = round ((0.1 * len(all_images)))

train_list = all_images[:train_size]
val_list   = all_images[train_size:train_size + val_size]
test_list  = all_images[train_size + val_size:]

# --- Fonction de copie image + label ---
def copy_pair(image_path, img_target, lbl_target):
    base, _ = os.path.splitext(os.path.basename(image_path))
    label_path = os.path.join(IMAGES_PATH, base + ".txt")

    # deplacer l’image
    shutil.move(image_path, img_target)

    # Déplacer le label si présent
    if os.path.exists(label_path):
        shutil.move(label_path, lbl_target)
    else:
        print(f"⚠ Aucun label trouvé pour {base}")

# --- Copie des fichiers ---
for img in train_list:
    copy_pair(img, train_images, train_labels)

for img in val_list:
    copy_pair(img, val_images,val_labels)

for img in test_list:
    copy_pair(img, test_images, test_labels)
 
    

print(f"Transfert terminé pour : {len(train_list)} images dans 'train_images', {len(val_list)} dans 'val_images', {len(test_list)} dans 'test'.")


### &emsp;&emsp;&emsp;&emsp;4.3) Structure definitive pour Pytorch </p>
on va maintenant créer la structure definitive des dossier 

In [ ]:
import os
data=os.path.join(WORK_PATH,"data")
data=data.replace("\\","/")
os.makedirs(data,exist_ok=True)
#on deplace tout dans le dossier data 
IMAGES_PATH=IMAGES_PATH.replace("\\","/")
LABELS_PATH=os.path.join(WORK_PATH,"labels")
LABELS_PATH=LABELS_PATH.replace("\\","/")

shutil.move(IMAGES_PATH, data)
shutil.move(LABELS_PATH, data)

# 5. lancement de l'entrainement
### &emsp;&emsp;&emsp;&emsp;5.1) Création du fichier yaml 

on va maintenant créer le fichier .yaml indispensable à pytorch pour l'entrainement et qui contient:</p>
. le chemin de vos images train, validation, et test</p>
. le nom de vos Classes</p>

    Vous devez remplacer :
    0: droite 
    1: gauche 
    etc.. 
    par le nom de vos classes et avec la même orthographe (majuscule et miniscule)


In [ ]:
yaml_content = f"""
path: {data}
train: {data}/images/train_images
val: {data}/images/val_images
test: {data}/image/test_images

names:
  0: droite
  1: gauche
  2: double
"""
yalm_path=os.path.join(data,"data.yaml")
with open(yalm_path, "w") as f:
    f.write(yaml_content)

print("Fichier data.yaml créé !")

### &emsp;&emsp;&emsp;&emsp;5.2) Import et configuration </p>

on va importer les bibliothèque nécessaires pour faire fonctionner Pytorch et verifier si </p>
l'accelération (cuda) graphique est possible

In [ ]:
# Cellule 1 : imports
from ultralytics import YOLO
import matplotlib.pyplot as plt
import os

# Optionnel : vérifier le GPU
import torch
print("PyTorch version:", torch.__version__)
print("CUDA dispo:", torch.cuda.is_available())

### &emsp;&emsp;&emsp;&emsp;5.3) chargement du modèle </p>
on va telecharger le modèle pré-entrainé sur lequel nous allons appliquer nos données

In [ ]:
#charger un modèle pré-entraîné (par ex. YOLOv8n)
#insister de temps en temps le téléchargement ne se fait pas 
model = YOLO("yolo11n.pt")  # nano, plus léger

### &emsp;&emsp;&emsp;&emsp;5.4) Lancement de l'entrainement et visualisation des courbes</p>
#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;A) les bibliothèques
nous allons intaller la bibliothèque panda pour la visialisation des courbes et lancer un mathplotlib en live</p>
pour suivre l'évolution de ces dernières au cours du temps 

In [ ]:
pip install pandas

In [ ]:
%matplotlib inline

#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;B) Préparation des graphiques 
La cellule suivante sert a preparer la visualisation des graphiques durant l'entrainement avec lisage des courbes </p> 
vous verez au fur et à mesure de l'apprentissage les courbes de loss, le mAP et les metrics de precision et recall

In [ ]:
#############################################################la ça marche mais pas de changement de results 
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

def smooth_curve(values, weight=0.85):
    smoothed = []
    last = values[0]
    for v in values:
        last = last * weight + (1 - weight) * v
        smoothed.append(last)
    return smoothed

def plot_dynamic_results(csv_path):
    df = pd.read_csv(csv_path)
    clear_output(wait=True)

    epochs = df["epoch"]

    def s(col):
        return smooth_curve(df[col].values) if col in df.columns else None

    # FIGURE 3x2
    fig, axes = plt.subplots(3, 2, figsize=(16,14))
    ax1, ax2, ax3, ax4, ax5, ax6 = axes.flatten()

    # 1) BOX LOSS
    if "train/box_loss" in df.columns:
        ax1.plot(epochs, s("train/box_loss"), label="Train Box Loss", color="blue")
    if "val/box_loss" in df.columns:
        ax1.plot(epochs, s("val/box_loss"), label="Val Box Loss", color="red")
    ax1.set_title("Box Loss")
    ax1.grid(True)
    ax1.legend()

    # 2) CLS LOSS
    if "train/cls_loss" in df.columns:
        ax2.plot(epochs, s("train/cls_loss"), label="Train Cls Loss", color="blue")
    if "val/cls_loss" in df.columns:
        ax2.plot(epochs, s("val/cls_loss"), label="Val Cls Loss", color="red")
    ax2.set_title("Classification Loss")
    ax2.grid(True)
    ax2.legend()

    # 3) DFL LOSS
    if "train/dfl_loss" in df.columns:
        ax3.plot(epochs, s("train/dfl_loss"), label="Train DFL Loss", color="blue")
    if "val/dfl_loss" in df.columns:
        ax3.plot(epochs, s("val/dfl_loss"), label="Val DFL Loss", color="red")
    ax3.set_title("DFL Loss")
    ax3.grid(True)
    ax3.legend()

    # 4) PRECISION / RECALL
    if "metrics/precision(B)" in df.columns:
        ax4.plot(epochs, s("metrics/precision(B)"), label="Precision", color="blue")
    if "metrics/recall(B)" in df.columns:
        ax4.plot(epochs, s("metrics/recall(B)"), label="Recall", color="red")
    ax4.set_title("Precision / Recall")
    ax4.grid(True)
    ax4.legend()

    # 5) mAP
    if "metrics/mAP50(B)" in df.columns:
        ax5.plot(epochs, s("metrics/mAP50(B)"), label="mAP50", color="blue")
    if "metrics/mAP50-95(B)" in df.columns:
        ax5.plot(epochs, s("metrics/mAP50-95(B)"), label="mAP50-95", color="red")
    ax5.set_title("mAP Metrics")
    ax5.grid(True)
    ax5.legend()

    # 6) vide
    ax6.axis("off")

    plt.tight_layout()
    display(fig)
    plt.close(fig)



#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;C) Entrainement et courbes en lives 
<div style="background-color:#fff3cd; padding:12px; border-left:6px solid #ffcc00;">
🚨 ⚠️ ‼️ 🔥 
Ici les choses vont se compliquer: </p>
Pour visualiser les courbes en directe durant l'entrainement nous devons absolument lancer la cellule d'entrainement dans un thread 
(execution séparé du notebook) </p> car sinon on ne pourras pas visualiser les graphique en temps réel durant l'apprentissage les cellules jupyter ne pouvant s'executer que l'une après l'autre </div>

>Mais problème, la visualisation en direct ne peux se produire que si vous avez lancer au moins une fois l'apprentissage (Bug Yolo), si c'est votre premier entrainement vous ne visualiserez vos courbes qu'a la fin de l'entrainement.</p>
  C'est pourquoi nous allons lancer dans un ordre précis qu'il faut absolument respecter :  
    . Un entrainement avec seulement 2 epochs (très rapides cellule ci-dessous)  
    . Nous lancerons la celules de visualisation juste après **(cellule D)**  
    . Nous effacerons les données **(cellule E)**  
    . Nous relancerons un entrainement correct nombre epochs à choisir **(cellule F)**  
    . Enfin durant l'entrainement nous relancerons **(cellule D)** pour visualiser les graphiques en live 


 

In [ ]:
from pathlib import Path
# entraînement
import threading
model = YOLO("yolo11n.pt")  # nano, plus léger
path_model=Path(os.path.join(WORK_PATH,"models"))

def train_yolo():
    global results
    results = model.train(
        data=f"{yalm_path}",  # chemin vers ton data.yaml
        epochs=2,
        imgsz=640,
        batch=16,
        project=path_model,
        name="yolo11n_custom",
        plots=True
   
)
    print("DONE_FIRST TRAINING FINISH")
# Lancer l'entraînement en arrière-plan
thread = threading.Thread(target=train_yolo)
thread.start()

#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;D) Visualisation des courbes 
lancer la cellule suivante directement après celle d'avant sans attendre de vois afficher  
DONE_FIRST TRAINING FINISH 

In [ ]:
###########################################la ça marche 
import time
from pathlib import Path

print("⏳ Attente du démarrage de l'entraînement...")

# 1) Attendre que le thread démarre réellement
while not thread.is_alive():
    time.sleep(0.5)

# 2) Attendre que 'results' soit créé
while "results" not in globals() or results is None:
    time.sleep(0.5)

# 3) Attendre que save_dir soit défini
while not hasattr(results, "save_dir") or results.save_dir is None:
    time.sleep(0.5)

csv_path = Path(results.save_dir) / "results.csv"

# 4) Attendre que le CSV soit créé
while not csv_path.exists():
    time.sleep(0.5)

print("📊 Visualisation dynamique en cours…")

# 5) Boucle dynamique
while True:
    plot_dynamic_results(csv_path)

    # Arrêt automatique quand l'entraînement est fini
    if not thread.is_alive():
        print("✔️ Entraînement terminé — arrêt de la visualisation.")
        break

    time.sleep(3)


#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;E) Effacement des datas du dossier entrainement 
après que les graphiques s'affiche dans la cellule au dessus, il faut maintenant supprimer les données en lancant la cellule ci dessous 

In [ ]:
import os
import shutil

path_delete = os.path.join(WORK_PATH, "models")
path_delete = path_delete.replace("\\", "/")

# Vérifier que le dossier existe
if os.path.isdir(path_delete):
    for item in os.listdir(path_delete):
        item_path = os.path.join(path_delete, item)
        item_path = item_path.replace("\\", "/")

        shutil.rmtree(item_path)
        print("anciennes données effacées, passer à la cellule suivante")
else:
    print("passer à la cellule suivante")

#### &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;F) entrainement réel 
Lancer la cellule suivante pour l'entrainement réel  
vous pouvez choisir ou changer le nombre d'epochs  
une fois lancer, relancer la **cellule D** pour visualiser vos graphique en LIVE  
ATTENTION UNE FOIS LA CELLULE LANCER VOUS DEVEZ ABSOLUMENT ATTENDRE LA FIN DE L4ENTRAINEMENT AVANT DE LANCER LES CELLULE SUIVANTES

In [ ]:
###### entrainement réel 
### from ultralytics import YOLO
import threading
from pathlib import Path

def train_yolo():
    global results
    print("➡️ train_yolo() démarré")

    try:
        model = YOLO("yolov8n.pt")
        results = model.train(
            data=yalm_path,
            epochs=500,
            imgsz=640,
            batch=16,
            workers=0, 
            project=path_model,
            name="yolo11n_custom",
            plots=True,
            save=True
            # patience=0,     # uncomment this line if you would not have auto stop 
            #save_period=1,   # <--- force l’écriture à chaque epoch

        )
        print("➡️ train_yolo() terminé")
    except Exception as e:
        print("🔥 ERREUR dans train_yolo() :", e)

thread = threading.Thread(target=train_yolo)
thread.start()

print("🚀 Thread lancé :", thread)


ATTENTION  IMPORTANT : 
La cellule suivante est obligatoire que si vous souhaitez faire un second entrainement  
a ce momment lancer la cellule puis **recommencer le jupyter notebook a partir du point 5.4**
Par contre si votre entrainement vous convient il ne faut surtout pas la lancer pour conserver tous vos poids, model, metric etc...

In [ ]:
import shutil
path_delete=os.path.join(WORK_PATH,"runs","models")
path_delete=path_delete.replace("\\","/")  

for item in os.listdir(path_delete):
    item_path=os.path.join(path_delete,item)
    item_path=item_path.replace("\\","/")  

    shutil.rmtree(item_path)

# 6. test du model sur la réalité terrain 
### &emsp;&emsp;&emsp;&emsp;6.1) récuperation du model
on va stocker dans une variable le dernier entrainement de votre model

In [ ]:
mon_model=Path(os.path.join(results.save_dir,"weights","best.pt"))
print(mon_model)

### &emsp;&emsp;&emsp;&emsp;6.2) lancement des variable pour dessiner les boites de detections
la cellule suivante correspond à des fonctions de mise en forme de vos :  
images test, des labels et du tracé des boites de detection sur vos images de tests

In [ ]:
def smooth_curve(values, weight=0.85):
    smoothed = []
    last = values[0]
    for v in values:
        last = last * weight + (1 - weight) * v
        smoothed.append(last)
    return smoothed

def load_yolo_labels(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            boxes.append([cls, xc, yc, w, h])
    return boxes

def yolo_to_xyxy(box, img_w, img_h):
    cls, xc, yc, w, h = box
    x1 = int((xc - w/2) * img_w)
    y1 = int((yc - h/2) * img_h)
    x2 = int((xc + w/2) * img_w)
    y2 = int((yc + h/2) * img_h)
    return cls, x1, y1, x2, y2

def draw_boxes(img, boxes, color, label_prefix="", class_names=None):
    img = img.copy()
    for box in boxes:
        if len(box) == 5:
            cls, x1, y1, x2, y2 = box
            conf = None
        else:
            cls, x1, y1, x2, y2, conf = box

        cls_name = class_names[int(cls)] if class_names else str(int(cls))

        label = f"{label_prefix}{cls_name}"
        if conf is not None:
            label += f" {conf:.2f}"

        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            img, label, (x1, max(0, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2
        )
    return img


In [ ]:
# Charger un fichier d’annotations YOLO (.txt) 
############################################################ancien a supprimer 
def load_yolo_labels(label_path: Path):
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path, "r") as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls, xc, yc, w, h = map(float, parts)
            boxes.append([cls, xc, yc, w, h])
    return boxes

# Convertir YOLO -> pixels
def yolo_to_xyxy(box, img_w, img_h):
    cls, xc, yc, w, h = box
    x1 = int((xc - w/2) * img_w)
    y1 = int((yc - h/2) * img_h)
    x2 = int((xc + w/2) * img_w)
    y2 = int((yc + h/2) * img_h)
    return cls, x1, y1, x2, y2

# Dessiner des boîtes
def draw_boxes(img, boxes, color, label_prefix="", class_names=None):
    img = img.copy()
    for box in boxes:
        if len(box)==5:
            cls, x1, y1, x2, y2 = box
            conf=None
        else: 
            cls, x1, y1, x2, y2, conf = box 
        #nom de la classe
        cls_name=class_names[int(cls)] if class_names else str(int(cls))
        label=f"{label_prefix}{cls_name}"
        if conf is not None:
            label +=f"{conf:.2f}"
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            img,
            label,
            (x1, max(0, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            color,
            2,
        )
    return img


### &emsp;&emsp;&emsp;&emsp;6.3) on va charger le modèle à appliquer
on va maintenant applique le modèle YOLO sur votre modèle

In [ ]:
model_sav = YOLO(mon_model)  # à adapter en fonction de votre nom
print("Modèle chargé.")


### &emsp;&emsp;&emsp;&emsp;6.4) déclaration des variables des images et labels test
on va stocker dans deux variables les chemins spécifiques des images à tester

In [ ]:
images_dir = Path(os.path.join(WORK_PATH,"data","images","test_images"))        # dossier contenant les .jpg
labels_dir = Path(os.path.join(WORK_PATH,"data","labels","test_images"))        # dossier contenant les .txt



#assert images_dir.exists(), "Dossier images introuvable"
#assert labels_dir.exists(), "Dossier annotations introuvable"

print("📂 Images :", images_dir)
print("📂 Annotations :", labels_dir)


### &emsp;&emsp;&emsp;&emsp;6.5) Application du modèle sur images test et comparaison réalité terrain 
cette cellule vous montrera à gauche la réalité terrain (c'est-à-dire) les annotations que vous avez fait sur vos immages test  
à droite les prédictions de detection de votre modèle avec le seil de confiance  
vous pouvez relancer cette cellule en changeant le seuil de confiance (CONF_THRESHOLD) si votre modèle detecte trop de faux positifs  
. seuil 0.3 à 0.5 (souvent nombreux faux positifs)  
. seuil 0.5 à 0.7 (le plus souvent utiliser en deep learning)  
. seuil 0.8 à 0.9 (souvent perte de positifs) 

In [ ]:
import cv2

CONF_THRESHOLD = 0.75  # vous pouvez changer ce seuil
for img_path in sorted(images_dir.glob("*.jpg")):

    # Chemin vers l’annotation correspondante
    label_path = labels_dir / (img_path.stem + ".txt")

    # 1) Charger l'image
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        print(f"⚠️ Impossible de lire l'image : {img_path}")
        continue
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # 2) Charger les annotations réelles (GT)
    gt_raw = load_yolo_labels(label_path)
    gt_boxes = [yolo_to_xyxy(b, w, h) for b in gt_raw]

    # 3) Prédiction du modèle
    results = model_sav(img, conf=CONF_THRESHOLD, verbose=False)[0]
    pred_boxes = []
    for b in results.boxes:
        cls = int(b.cls)
        conf=float(b.conf)
        x1, y1, x2, y2 = map(int, b.xyxy[0])
        pred_boxes.append([cls, x1, y1, x2, y2, conf])

    # 4) Dessiner GT (vert) et prédictions (rouge)
    img_gt = draw_boxes(img, gt_boxes, (0, 255, 0), "GT ",model.names)
    img_pred = draw_boxes(img, pred_boxes, (255,255, 0), "PRED ",model.names)

    # 5) Affichage côte à côte
    plt.figure(figsize=(14, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(img_gt)
    plt.title(f"Réalité terrain (GT)\n{img_path.name}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(img_pred)
    plt.title("Prédiction modèle")
    plt.axis("off")

    plt.show()


# 7. Sauvegarde du modele

In [ ]:
print (mon_model)
# 1. Création du dossier de sauvegarde
sav_final = Path(WORK_PATH) / "model_saving"
sav_final.mkdir(parents=True, exist_ok=True)

print("Dossier créé :", sav_final)

# 3. Chemin du fichier exporté par YOLO
source = mon_model

# 4. Chemin final avec ton nom
destination = sav_final / "brice.pt" 

# 5. copie + renommage
shutil.copy(str(source), str(destination))

print("Modèle exporté vers :", destination)

In [ ]:
saving_model=destination

final_model=YOLO(saving_model)
print("model charger et pret à l'export")

final_model.export(format="ncnn")

In [ ]:
############################################################pas besoin pour l'instant
print (mon_model)
# 1. Création du dossier de sauvegarde
sav_final = Path(WORK_PATH) / "model_saving"
sav_final.mkdir(parents=True, exist_ok=True)

print("Dossier créé :", sav_final)

# 2. Export YOLO (ne respecte pas save_dir)
model_sav.export(format="ncnn")

# 3. Chemin du fichier exporté par YOLO
source = mon_model

# 4. Chemin final avec ton nom
destination = sav_final / "brice.ncnn"

# 5. copie + renommage
shutil.copy(str(source), str(destination))

print("Modèle exporté vers :", destination)


# 8. Déploiement sur raspberry
copier le fichier du dossier model_saving sur une clef et le copier ensuite sur une raspberry